# 3. Advanced concepts

Seven things you can use, each replacing machinery you would otherwise write
yourself.

1. **residues** without a contour integral
2. **root multiplicity** in one read, with no threshold
3. **limits at infinity** by evaluating there
4. **branch points** carrying their own exponent
5. **the log axis**, for divergences slower than any power
6. **a divergent series** turned into a number, with its ambiguity
7. **measurement uncertainty** budgets, and the decision the GUM leaves open

Then the one pattern where the alternatives do not work, and a
list of what this does not do.

In [1]:
import math, warnings
import composite.composite_lib as cl
import composite.composite_extended as ce
from composite import (R, ZERO, INF, Composite, sin, cos, exp, ln, sqrt,
                       explain, budget, Quantity, montecarlo)
from composite.resummation import resum_median, flat_term, classify_singularity
from composite.backends.config import use_dict

use_dict(); cl._refresh_constants()
# MAX_ACTIVE_DIMS is left at its default of 60. Raising it removes the
# truncation guard, and an iterative algorithm then grows terms without
# bound -- solve_ode below did not finish 20 steps in 30s at 10**9, and
# finishes 200 steps in 0.86s at the default.
warnings.simplefilter("ignore")
for w in (cl.CancellationWarning, cl.NotConventionalWarning):
    warnings.filterwarnings("ignore", category=w)
print("ready")

ready


## 1. Residues, by evaluating at the pole

A residue is the coefficient of `(z-a)**-1`, which in this system is simply the
coefficient at grade `+1`. Seed the pole and read it off - no contour, no
derivative formula for higher-order poles.

In [2]:
poles = [("1/(z-2)                at 2", lambda z: R(1) / (z - R(2)),                2.0, 1.0),
         ("exp(z)/z                at 0", lambda z: exp(z) / z,                      0.0, 1.0),
         ("z/((z-1)(z-3))          at 3", lambda z: z / ((z - R(1)) * (z - R(3))),   3.0, 1.5),
         ("z/((z-1)(z-3))          at 1", lambda z: z / ((z - R(1)) * (z - R(3))),   1.0, -0.5),
         ("1/(z-2)**3              at 2", lambda z: R(1) / (z - R(2)) ** 3,          2.0, 0.0)]
print("%-32s %-12s %-10s %s" % ("function", "residue", "want", "pole order"))
for label, f, at, want in poles:
    print("%-32s %-12.6g %-10g %s" % (label, ce.residue(f, at), want, ce.pole_order(f, at)))

function                         residue      want       pole order
1/(z-2)                at 2      1            1          1
exp(z)/z                at 0     1            1          1
z/((z-1)(z-3))          at 3     1.5          1.5        1
z/((z-1)(z-3))          at 1     -0.5         -0.5       1
1/(z-2)**3              at 2     0            0          3


The last row is the useful one: a third-order pole has residue 0, and `pole_order`
tells you the 3 that the classical formula needs *before* you can apply it.

## 2. Root multiplicity, with nothing to threshold

Is this root simple, double or triple? Classically you compare `f`, `f'`, `f''`
against a tolerance and hope. Here the leading grade **is** the multiplicity, and
it is an integer.

In [3]:
for n in (1, 2, 3, 4):
    y = (R(2) + ZERO - R(2)) ** n          # (x - 2)**n at x = 2
    print("(x-2)**%d  ->  %-30s multiplicity %s" % (n, y, y.lead_order()))
print()
print("and it survives being mixed into something larger:")
x = R(2) + ZERO
mixed = (x - R(2)) ** 3 * (x + R(5))
print("  (x-2)**3 * (x+5) at 2 ->", mixed, " multiplicity", mixed.lead_order())

(x-2)**1  ->  <|0|₀ |1|₋₁>                   multiplicity 1.0
(x-2)**2  ->  <|0|₀ |0|₋₁ |1|₋₂>             multiplicity 2.0
(x-2)**3  ->  <|0|₀ |0|₋₁ |0|₋₂ |1|₋₃>       multiplicity 3.0
(x-2)**4  ->  <|0|₀ |0|₋₁ |0|₋₂ |0|₋₃ |1|₋₄> multiplicity 4.0

and it survives being mixed into something larger:
  (x-2)**3 * (x+5) at 2 -> <|0|₀ |0|₋₁ |0|₋₂ |7|₋₃ |1|₋₄>  multiplicity 3.0


The same read answers four classically separate questions: multiplicity of a root,
order of contact between two curves, corank of a matrix, and how thoroughly a
quantity is zero. One subtraction, one integer, no tolerance.

## 3. Limits at infinity

Positive grades are where unboundedness lives, so you can evaluate *at* infinity
rather than taking a limit towards it.

In [4]:
limits = [("(3x**2 + 1)/(x**2 - 5)", lambda x: (R(3) * x * x + R(1)) / (x * x - R(5)), 3.0),
          ("(x + 1)/(2x + 7)",       lambda x: (x + R(1)) / (R(2) * x + R(7)),           0.5),
          ("(x**2 + 1)/(x**3 - 1)",  lambda x: (x * x + R(1)) / (x ** 3 - R(1)),         0.0)]
for label, f, want in limits:
    print("%-26s -> %-10.6g want %g" % (label, ce.limit_at_infinity(f), want))

(3x**2 + 1)/(x**2 - 5)     -> 3          want 3
(x + 1)/(2x + 7)           -> 0.5        want 0.5
(x**2 + 1)/(x**3 - 1)      -> 0          want 0


## 4. Branch points carry their exponent

`sqrt(h)` is not a power series in `h`. It sits at grade `-1/2`, and the half is the
information: this is a branch point, not a pole and not a zero.

In [5]:
from fractions import Fraction

for label, v in (("sqrt(h)",          sqrt(ZERO)),
                ("h**Fraction(1,2)", ZERO ** Fraction(1, 2)),
                ("h**Fraction(1,3)", ZERO ** Fraction(1, 3)),
                ("h**Fraction(3,2)", ZERO ** Fraction(3, 2)),
                ("sqrt(sqrt(h))",    sqrt(sqrt(ZERO))),
                ("1/sqrt(h)",        R(1) / sqrt(ZERO))):
    print("%-18s %-20s lead_order %s" % (label, v, v.lead_order()))

print()
print("A float exponent is fine when the float is exact: 1.5 is 3/2 in binary.")
print("  ZERO ** 1.5           ->", ZERO ** 1.5)
print("  ZERO ** Fraction(3,2) ->", ZERO ** Fraction(3, 2))
print()
print("A third is not exact in float64, as an exponent or as a grade, so grades that")
print("are thirds cannot be combined here; the library refuses rather than rounds:")
try:
    (ZERO ** Fraction(1, 3)) ** 3
except Exception as problem:
    print("  (ZERO ** Fraction(1,3)) ** 3 ->", type(problem).__name__)
from composite.backends.config import use_fractional_dict
use_fractional_dict(); cl._refresh_constants()
print("  on the fractional backend, whose grades are exact rationals:")
print("  (ZERO ** Fraction(1,3)) ** 3 ->", (cl.ZERO ** Fraction(1, 3)) ** 3, "  == h:", (cl.ZERO ** Fraction(1, 3)) ** 3 == cl.ZERO)
use_dict(); cl._refresh_constants()

print()
print(explain(lambda x: sqrt(x), 0.0))

sqrt(h)            <1_-0.5>             lead_order 0.5
h**Fraction(1,2)   <1_-0.5>             lead_order 0.5
h**Fraction(1,3)   <1_-0.333333>        lead_order 0.3333333333333333
h**Fraction(3,2)   <1_-1.5>             lead_order 1.5
sqrt(sqrt(h))      <1_-0.25>            lead_order 0.25
1/sqrt(h)          <1_0.5>              lead_order -0.5

A float exponent is fine when the float is exact: 1.5 is 3/2 in binary.
  ZERO ** 1.5           -> <1_-1.5>
  ZERO ** Fraction(3,2) -> <1_-1.5>

A third is not exact in float64, as an exponent or as a grade, so grades that
are thirds cannot be combined here; the library refuses rather than rounds:
  (ZERO ** Fraction(1,3)) ** 3 -> InexactGradeError
  on the fractional backend, whose grades are exact rationals:
  (ZERO ** Fraction(1,3)) ** 3 -> <|1|₋₁>   == h: True

f at x = 0: value 0, but with a corner: it leaves the point like x**0.5, so the slope is infinite


In [6]:
sqrt(ZERO) + ZERO     # a half grade beside an integer one, in one number

grade,coefficient,meaning
-0.5,1,"branch point, order 0.5"
-1,1,order 1 term


Note the table does **not** offer `d(1)` beside a branch point. With a half grade
present `coefficient * n!` is not a derivative, and the display says so rather than
printing a plausible number.

## 5. The log axis

`ln(1/h)` grows without bound but slower than any power of `1/h`, so it cannot live
on the power axis. It gets its own: a **vector grade**, `(power, log)`.

In [7]:
for label, v in (("ln(h)",         ln(ZERO)),
                ("ln(1/h)",       ln(R(1) / ZERO)),
                ("ln(ln(1/h))",   ln(ln(R(1) / ZERO))),
                ("1/h + ln(1/h)", R(1) / ZERO + ln(R(1) / ZERO))):
    print("%-16s %-22s lead_dim %s" % (label, v, v.lead_dim()))

ln(h)            <-1_(0,1)>             lead_dim (0, 1)
ln(1/h)          <1_(0,1)>              lead_dim (0, 1)
ln(ln(1/h))      <1_(0,0,1)>            lead_dim (0, 0, 1)
1/h + ln(1/h)    <1_1 1_(0,1)>          lead_dim 1


In [8]:
R(1) / ZERO + ln(R(1) / ZERO)    # a pole and a log term, two axes, one number

grade,coefficient,meaning
1,1,"unbounded, order 1"
"(0, 1)",1,"log axis, depth 1"


In [9]:
print("ordering still works, a power beats a logarithm:")
print("  1/h > ln(1/h) ?", (R(1) / ZERO) > ln(R(1) / ZERO))
print()
print("and each projects to float correctly:")
print("  ln(1/h) ->", ln(R(1) / ZERO).to_ieee754())
print("  ln(h)   ->", ln(ZERO).to_ieee754(), " (ln of a small positive is large NEGATIVE)")
print()
print("explain names the three log cases apart:")
for label, f in (("ln(x)        at 0", lambda x: ln(x)),
                 ("x*ln(x)      at 0", lambda x: x * ln(x)),
                 ("1/ln(x)      at 0", lambda x: R(1) / ln(x))):
    print("  %-20s kind = %s" % (label, explain(f, 0.0).kind))

ordering still works, a power beats a logarithm:
  1/h > ln(1/h) ? True

and each projects to float correctly:
  ln(1/h) -> inf
  ln(h)   -> -inf  (ln of a small positive is large NEGATIVE)

explain names the three log cases apart:
  ln(x)        at 0    kind = log-unbounded
  x*ln(x)      at 0    kind = steep
  1/ln(x)      at 0    kind = log-value


## 6. A divergent series, turned into a number

`sum (-1)**n n! g**n` diverges for every `g`. Its Borel sum is the standard model
for a **renormalon**: for `g < 0` the Borel singularity sits on the integration
contour, so the answer carries an irreducible ambiguity.

Both come from the coefficients alone. Checked against closed forms.

In [10]:
from mpmath import mp, mpf, ei, exp as mexp, pi as mpi
mp.dps = 30
euler = [((-1) ** n) * math.factorial(n) for n in range(24)]

print("%-7s %-24s %-24s %-13s %s" % ("g", "resum_median", "closed form", "flat_term", "pi e^(1/g)/|g|"))
for g in (-0.05, -0.1, -0.2):
    a = -g
    pv  = float(mexp(-1 / mpf(a)) / mpf(a) * ei(1 / mpf(a)))
    amb = float(mpi * mexp(-1 / mpf(a)) / mpf(a))
    med, _ = resum_median(euler, g)
    _, flat = flat_term(euler, g)
    print("%-7.2f %-24.15g %-24.15g %-13.6g %-13.6g" % (g, med.real, pv, flat, amb))

g       resum_median             closed form              flat_term     pi e^(1/g)/|g|


-0.05   1.05595590559296         1.05595590559296         1.29506e-07   1.29506e-07  
-0.10   1.13147020473411         1.13147020473411         0.00142628    0.00142628   
-0.20   1.35383127745529         1.35383127745529         0.105839      0.105839     


Fifteen digits on the value, six on the ambiguity, with no contour chosen by hand.

A **real** renormalon is a branch point rather than a pole, and there the library
refuses to report an ambiguity instead of inventing one.

In [11]:
def branch_point_series(n):
    """Borel transform (1+u)**(-1/2): a cut, which is what a renormalon is."""
    out, double_factorial = [], 1.0
    for k in range(n):
        out.append(((-1) ** k) * double_factorial / 2.0 ** k)
        double_factorial *= (2 * k + 1)
    return out

cut = branch_point_series(26)
kind, where, residue, why = classify_singularity(cut)
print("classify_singularity ->", kind, "at %.6f  (true location -1)" % where)
print("  residue            ->", residue, " <- a cut has none")
print("  why                ->", why["why"])
print()
print("the VALUE still comes out:", "%.12g" % resum_median(cut, -0.1)[0].real,
      " (40-digit quadrature 1.06075161986)")
try:
    flat_term(cut, -0.1)
except NotImplementedError as problem:
    print()
    print("but flat_term REFUSES:")
    print(" ", str(problem).split(".")[0] + ".")

classify_singularity -> cut at -1.004379  (true location -1)
  residue            -> None  <- a cut has none
  why                -> roots clustered along a ray (gap/|z0| 0.036); root not converging (rate 0.144); residue sliding (1.75e-02 per order)



the VALUE still comes out: 1.06075161986  (40-digit quadrature 1.06075161986)

but flat_term REFUSES:
  the Borel transform has a BRANCH POINT at np.


That refusal is worth more than a number would be. The *action* of a branch-point
renormalon is determined; its *normalisation* is not a residue, and returning one
would be a number with no meaning attached.

## 7. Measurement uncertainty, and the decision the GUM leaves open

First-order uncertainty propagation is well served elsewhere. What is here is the
part the GUM leaves to the engineer: the **bias**, the higher-order variance, and
whether the linearised budget is admissible at all or whether Supplement 1 requires
a Monte Carlo run instead.

In [12]:
model = lambda length, width: length * width
inputs = {"length": Quantity(2.0, 0.1, unit="m"),
          "width":  Quantity(3.0, 0.2, unit="m")}
b = budget(model, inputs)
print("value            %.6g" % b.value)
print("u (linear)       %.6g" % b.u_linear)
print("u (higher order) %.6g" % b.u_higher)
print("bias             %.6g" % b.bias)
print("verdict          %s" % b.verdict)
for reason in b.reasons:
    print("  -", reason)

value            6
u (linear)       0.5
u (higher order) 0.5004
bias             0
verdict          HIGHER-ORDER TERMS NEEDED
  - coverage endpoints move by 0.0274 / 0.0271, tolerance 0.005
  - output skewness +0.115; the interval is asymmetric


In [13]:
print(b.table())

input  estimate      u(x)         c = df/dx      c*u(x)        index%   dist
----------------------------------------------------------------------------
width  3             0.2          2              4.0000e-01    64.00    normal
length 2             0.1          3              3.0000e-01    36.00    normal
----------------------------------------------------------------------------
value  6
u_c (linear) 5.000000e-01
u_c (higher) 5.003998e-01
bias   +0.000000e+00
skewness +0.114924
ex. kurtosis +0
95% (k=1.960) [5.020018008, 6.979981992]
95% corrected [5.047434131, 7.007034728]
verdict HIGHER-ORDER TERMS NEEDED
         - coverage endpoints move by 0.0274 / 0.0271, tolerance 0.005
         - output skewness +0.115; the interval is asymmetric
fd cross-check worst 1.31e-11 on 'length'


The verdict is the deliverable: it tells you when the linear budget is not good
enough, which is a judgement the standard requires and most tools do not make.

## The one pattern with no real alternative

A comparison ties at the top order, and is decided by what sits one grade down.
Three independent uses, in three unrelated fields.

**Simplex pivoting.** Beale's 1955 example cycles forever under Dantzig's rule
because every minimum-ratio test ties at 0. Perturb the right-hand side by `h**i`
and the ties break.

In [14]:
print("float ratio test  :", [0.0 / 0.25, 0.0 / 0.50], " tied, choose arbitrarily, cycle")

b0 = Composite({0: 0.0, -1: 1.0})      # 0 + h
b1 = Composite({0: 0.0, -2: 1.0})      # 0 + h**2
ratios = [b0 / 0.25, b1 / 0.50]
print("graded ratio test :", [str(r) for r in ratios])
print("                    4h vs 2h**2; grade -2 is deeper, so row 1 is SMALLER")
print("winner            :", "row 1" if ratios[1] < ratios[0] else "row 0")

float ratio test  : [0.0, 0.0]  tied, choose arbitrarily, cycle
graded ratio test : ['<|0|₀ |4|₋₁>', '<|0|₀ |2|₋₂>']
                    4h vs 2h**2; grade -2 is deeper, so row 1 is SMALLER
winner            : row 1


**Lexicographic probability.** Selten's trembling-hand perfect equilibrium is
defined by a limit, because at the limit every tremble is 0 and the comparison that
did the work has vanished. Here it stays in the number.

In [15]:
NOTHING, tremble = Composite({}), ZERO
# zero payoffs are SKIPPED, not added: a written zero would convert and be
# mistaken for the tremble itself
E_L = tremble * R(1)      # only the trembled action pays
E_R = NOTHING             # nothing pays
print("E[L] =", E_L, "  E[R] =", E_R, "  E[L] > E[R] ?", E_L > E_R)
print()
print("classically both payoffs are 0.0 and no arithmetic can prefer one.")

E[L] = <|1|₋₁>   E[R] = ∅   E[L] > E[R] ? True

classically both payoffs are 0.0 and no arithmetic can prefer one.


**Exact geometric predicates** are the third: a degenerate Delaunay test decided by
an infinitesimal that stays inside the arithmetic.

What the three share is the alternative: a hand-rolled symbolic comparator, or a
numeric `eps` that cannot scale. A float `eps` must satisfy `eps**m > 1e-16` so the
deepest perturbation survives **and** `eps << 1` so the answer is not moved. By
eight rows those conflict.

In [16]:
print("%-8s %-20s %s" % ("rows m", "smallest usable eps", "still a perturbation?"))
for m in (3, 5, 8, 16, 1000):
    smallest = 10 ** (-16.0 / m)
    verdict = ("yes" if smallest < 1e-3 else
               "marginal" if smallest < 1e-2 else "NO, it changes the problem")
    print("%-8d %-20.3g %s" % (m, smallest, verdict))

rows m   smallest usable eps  still a perturbation?
3        4.64e-06             yes
5        0.000631             yes
8        0.01                 NO, it changes the problem
16       0.1                  NO, it changes the problem
1000     0.964                NO, it changes the problem


## What this does not do

* **Coefficients are float64.** The grade axis extends the *exponent* range without
  bound and leaves coefficient precision exactly where it was.
* **No units.** The dimensional algebra is present - multiplication adds exponents,
  addition refuses to mix - but no basis is populated, and `a - a = a*h` is
  dimensionally closed only when the infinitesimal is dimensionless.
* **No complex numbers.** `i` needs a *twisted* axis, not a graded one; see
  `docs/Complex Numbers - Exploration (DRAFT).md`.
* **Not fast.** `jet` matches the grades on accuracy and is 7-55% quicker. The lanes
  are the differentiator, not the arithmetic.

In [17]:
print("the float64 ceiling, shown rather than asserted:")
print("  1.4884e38 - 15625 == 1.4884e38 ?", (1.4884e38 - 15625) == 1.4884e38)
print("  a 34-order cancellation cannot be WRITTEN in one float")
print()
print("but two scales on two grades is exact -- which is often the real fix:")
planck, bare = R(1.0), R(-1.0) + ZERO        # in units of M_Pl**2
total = planck + bare
print("  M_Pl**2 + m_bare**2 =", total, " grade -1 coefficient", total.coeffs_dict().get(-1))

the float64 ceiling, shown rather than asserted:
  1.4884e38 - 15625 == 1.4884e38 ? True
  a 34-order cancellation cannot be WRITTEN in one float

but two scales on two grades is exact -- which is often the real fix:
  M_Pl**2 + m_bare**2 = <|0|₀ |1|₋₁>  grade -1 coefficient 1.0


## Where to go next

| | |
|---|---|
| `demos/composite_simplex_pivoting.py` | cycling, and the perturbation that stops it |
| `demos/composite_lexicographic.py` | trembling-hand perfection without the limit |
| `demos/composite_basel.py` | Euler's Basel problem, poles named by order |
| `demos/composite_relativity.py` | the relativistic series, grade by grade |
| `docs/API Reference.md` | every call, with the costs stated |
| `docs/Zero Rules v2` | the rules, and what they give up |